# PCA vs SVD — eigenfaces on the class dataset

two ways to compute the same thing:

| path | operation | core function |
|------|-----------|---------------|
| **PCA** | eigendecompose the covariance matrix $X^\top X$ | `np.linalg.eigh` |
| **SVD** | directly decompose $X = U \Sigma V^\top$ | `np.linalg.svd` |

both produce the same eigenfaces. the goal here is to show *why* they're equivalent, verify it numerically, and then measure where they actually differ (reconstruction quality, recognition accuracy, compute time).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import timeit
from PIL import Image

IMG_SIZE   = (64, 64)
DATASET_PATH = "class_pictures"

def load_dataset(root):
    X, y, label_map = [], [], {}
    for idx, person in enumerate(sorted(os.listdir(root))):
        person_dir = os.path.join(root, person)
        if not os.path.isdir(person_dir):
            continue
        label_map[idx] = person
        for fname in sorted(os.listdir(person_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img = Image.open(os.path.join(person_dir, fname)).convert("L")
            img = img.resize(IMG_SIZE)
            X.append(np.array(img).flatten() / 255.0)
            y.append(idx)
    return np.array(X), np.array(y), label_map

X, y, label_map = load_dataset(DATASET_PATH)
print(f"loaded: {X.shape}  |  people: {len(label_map)}")

## 1. split + preprocessing

leave-one-out: 2 images per person → training (52 total), 1 image → test (26 total).

In [ ]:
train_idx, test_idx = [], []
for person_id in np.unique(y):
    idxs = np.where(y == person_id)[0]
    train_idx.extend(idxs[:-1])
    test_idx.append(idxs[-1])

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

mean_face  = X_train.mean(axis=0)
X_c        = X_train - mean_face   # centered training data
X_test_c   = X_test  - mean_face

n, d = X_c.shape
print(f"training: {X_c.shape}  |  test: {X_test_c.shape}  |  n={n}, d={d}")

## 2. the math — why they give the same answer

### PCA path

the covariance matrix is $C = \frac{1}{n-1} X_c^\top X_c$ — shape $(d \times d)$, far too large to form directly ($4096 \times 4096$).

**compact trick:** since $n \ll d$, compute the small version instead:
$$C_{\text{small}} = \frac{1}{n-1} X_c X_c^\top \quad (n \times n)$$

if $C_{\text{small}} v_i = \lambda_i v_i$, then the full-space eigenvectors are $u_i = X_c^\top v_i$ (normalized). those are the eigenfaces.

### SVD path

decompose $X_c$ directly:
$$X_c = U \Sigma V^\top \quad U: (n{\times}n),\; \Sigma: (n{\times}n),\; V^\top: (n{\times}d)$$

the **rows of $V^\top$** are the eigenfaces (right singular vectors).

### the link

$$X_c^\top X_c = V \Sigma^2 V^\top \quad \Longrightarrow \quad \lambda_i = \frac{\sigma_i^2}{n-1}$$

so $V$ from SVD **is** the eigenvector matrix of $X_c^\top X_c$. same result, different route.

## 3. compute eigenfaces — both methods

In [ ]:
# --- PCA via eigendecomposition ---
C_small      = X_c @ X_c.T / (n - 1)                    # (52, 52)
eigvals, evecs = np.linalg.eigh(C_small)                 # ascending order

order        = np.argsort(eigvals)[::-1]                 # sort descending
eigvals      = eigvals[order]
evecs        = evecs[:, order]

ef_full      = X_c.T @ evecs                             # back-project: (4096, 52)
eigenfaces_pca = (ef_full / np.linalg.norm(ef_full, axis=0)).T  # (52, 4096) normalized

explained_pca = eigvals / eigvals.sum()
print(f"PCA eigenfaces: {eigenfaces_pca.shape}")
print(f"top-5 explained variance: {explained_pca[:5].round(3)}")

In [ ]:
# --- SVD ---
U, S, Vt     = np.linalg.svd(X_c, full_matrices=False)  # Vt: (52, 4096)
eigenfaces_svd = Vt                                       # rows = right singular vectors

explained_svd = (S**2) / (S**2).sum()
print(f"SVD eigenfaces: {eigenfaces_svd.shape}")
print(f"top-5 explained variance: {explained_svd[:5].round(3)}")

## 4. are they the same?

eigenfaces are only defined up to a sign flip — the eigenvector $v$ and $-v$ are both valid. we align signs before comparing.

In [ ]:
def align_signs(ref, target):
    """flip rows of target to match the sign of ref"""
    signs = np.sign((ref * target).sum(axis=1))
    signs[signs == 0] = 1
    return target * signs[:, np.newaxis]

eigenfaces_pca_aligned = align_signs(eigenfaces_svd, eigenfaces_pca)

# absolute dot products between corresponding components (should be ≈ 1.0)
dots = np.abs(np.einsum("ij,ij->i", eigenfaces_pca_aligned, eigenfaces_svd))
print("abs dot product  |PCA · SVD|  for each component (1.0 = identical):")
for i in range(len(dots)):
    bar = "█" * int(dots[i] * 20)
    print(f"  PC{i+1:02d}  {dots[i]:.8f}  {bar}")

In [ ]:
max_diff = np.max(np.abs(eigenfaces_pca_aligned - eigenfaces_svd))
print(f"max absolute difference between all eigenface values: {max_diff:.2e}")
print("(numerical noise only — same result up to floating-point precision)")

In [ ]:
K_SHOW = 8
fig, axes = plt.subplots(2, K_SHOW, figsize=(16, 4))

for i in range(K_SHOW):
    axes[0, i].imshow(eigenfaces_pca_aligned[i].reshape(64, 64), cmap="gray")
    axes[0, i].set_title(f"PC {i+1}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(eigenfaces_svd[i].reshape(64, 64), cmap="gray")
    axes[1, i].set_title(f"SV {i+1}", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("PCA", fontsize=11)
axes[1, 0].set_ylabel("SVD", fontsize=11)
plt.suptitle("top 8 eigenfaces — PCA (top) vs SVD (bottom)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. reconstruction quality

if the eigenfaces are the same, reconstruction error should be identical at every $k$.

In [ ]:
def reconstruct(X_c_batch, components_k, mean):
    w = X_c_batch @ components_k.T
    return w @ components_k + mean

k_values = list(range(1, n + 1))
mse_pca, mse_svd = [], []

for k in k_values:
    rec_pca = reconstruct(X_test_c, eigenfaces_pca[:k], mean_face)
    rec_svd = reconstruct(X_test_c, eigenfaces_svd[:k], mean_face)
    mse_pca.append(np.mean((X_test - rec_pca) ** 2))
    mse_svd.append(np.mean((X_test - rec_svd) ** 2))

plt.figure(figsize=(9, 4))
plt.plot(k_values, mse_pca, label="PCA (eig)", linewidth=2)
plt.plot(k_values, mse_svd, label="SVD", linewidth=2, linestyle="--")
plt.xlabel("k (number of components)")
plt.ylabel("mean squared error")
plt.title("reconstruction error vs k — PCA vs SVD")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

max_recon_diff = max(abs(a - b) for a, b in zip(mse_pca, mse_svd))
print(f"max difference in MSE between methods across all k: {max_recon_diff:.2e}")

### visual reconstruction comparison at fixed k values

In [ ]:
k_show  = [3, 8, 15, 26]
subject = 0  # first test image
query_c = X_test_c[[subject]]

n_rows = 3  # original, PCA, SVD
n_cols = len(k_show) + 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 6))

row_labels = ["original", "PCA", "SVD"]
for r in range(n_rows):
    axes[r, 0].imshow(X_test[subject].reshape(64, 64), cmap="gray")
    axes[r, 0].set_title("original" if r == 0 else "", fontsize=9)
    axes[r, 0].set_ylabel(row_labels[r], fontsize=10)
    axes[r, 0].axis("off")

for j, k_val in enumerate(k_show, start=1):
    rec_p = reconstruct(query_c, eigenfaces_pca[:k_val], mean_face)
    rec_s = reconstruct(query_c, eigenfaces_svd[:k_val], mean_face)

    axes[0, j].imshow(X_test[subject].reshape(64, 64), cmap="gray")
    axes[0, j].set_title(f"k={k_val}", fontsize=9)
    axes[0, j].axis("off")

    axes[1, j].imshow(rec_p.reshape(64, 64), cmap="gray")
    axes[1, j].axis("off")

    axes[2, j].imshow(rec_s.reshape(64, 64), cmap="gray")
    axes[2, j].axis("off")

plt.suptitle(f"reconstructions — {label_map[y_test[subject]]}", fontsize=12)
plt.tight_layout()
plt.show()

## 6. recognition accuracy

sweep $k$ from 1 to 26 — nearest-neighbor recognition in eigenspace.

In [ ]:
def accuracy_at_k(eigenfaces_k):
    train_w = X_c @ eigenfaces_k.T
    correct = 0
    for i in range(len(X_test_c)):
        dists = np.linalg.norm(train_w - (X_test_c[[i]] @ eigenfaces_k.T), axis=1)
        if y_train[np.argmin(dists)] == y_test[i]:
            correct += 1
    return correct / len(y_test)

acc_pca = [accuracy_at_k(eigenfaces_pca[:k]) for k in k_values]
acc_svd = [accuracy_at_k(eigenfaces_svd[:k]) for k in k_values]

plt.figure(figsize=(9, 4))
plt.plot(k_values, acc_pca, label="PCA (eig)", linewidth=2)
plt.plot(k_values, acc_svd, label="SVD", linewidth=2, linestyle="--")
plt.xlabel("k (number of components)")
plt.ylabel("recognition accuracy")
plt.title("accuracy vs k — PCA vs SVD")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

best_k_pca = k_values[int(np.argmax(acc_pca))]
best_k_svd = k_values[int(np.argmax(acc_svd))]
print(f"best accuracy — PCA: {max(acc_pca):.2%} at k={best_k_pca}")
print(f"best accuracy — SVD: {max(acc_svd):.2%} at k={best_k_svd}")

## 7. compute time

measuring the time to compute all eigenfaces from scratch, averaged over 200 runs.

In [ ]:
N_RUNS = 200

def run_pca_eig():
    C = X_c @ X_c.T / (n - 1)
    eigvals, evecs = np.linalg.eigh(C)
    order = np.argsort(eigvals)[::-1]
    evecs = evecs[:, order]
    ef = X_c.T @ evecs
    return (ef / np.linalg.norm(ef, axis=0)).T

def run_svd():
    _, _, Vt = np.linalg.svd(X_c, full_matrices=False)
    return Vt

t_pca_ms = timeit.timeit(run_pca_eig, number=N_RUNS) / N_RUNS * 1000
t_svd_ms = timeit.timeit(run_svd,     number=N_RUNS) / N_RUNS * 1000

print(f"PCA (eig):  {t_pca_ms:.3f} ms  (avg over {N_RUNS} runs)")
print(f"SVD:        {t_svd_ms:.3f} ms  (avg over {N_RUNS} runs)")
print(f"ratio:      SVD is {t_svd_ms/t_pca_ms:.2f}x the cost of PCA on this dataset")
print()
print("note: on small n (52 samples), PCA's compact trick (n×n matrix) is fast.")
print("as n grows, SVD scales better — no need to form or solve a covariance matrix.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(["PCA (eig)", "SVD"], [t_pca_ms, t_svd_ms], color=["steelblue", "coral"], width=0.4)
ax.bar_label(bars, fmt="%.3f ms", padding=3, fontsize=10)
ax.set_ylabel("time (ms)")
ax.set_title(f"compute time — avg over {N_RUNS} runs\n(class dataset, n=52, d=4096)")
ax.set_ylim(0, max(t_pca_ms, t_svd_ms) * 1.4)
plt.tight_layout()
plt.show()

## 8. verdict

| | PCA (eigendecomposition) | SVD |
|---|---|---|
| **eigenfaces identical?** | ✓ yes (up to sign) | ✓ yes (up to sign) |
| **reconstruction error** | identical | identical |
| **recognition accuracy** | identical | identical |
| **intermediate matrix** | $n \times n$ covariance | none |
| **numerical stability** | loses precision squaring $X$ | more stable |
| **speed (small n)** | fast (compact trick) | comparable |
| **used by sklearn PCA** | no | **yes** — it uses SVD internally |

### key takeaway

PCA and SVD are two routes to the same destination. the eigenfaces, reconstruction quality, and recognition accuracy are **numerically identical**. the real difference is:

- **PCA** requires forming a covariance matrix, which squares the condition number and can lose precision
- **SVD** works directly on the data matrix — more stable and the method sklearn actually uses under the hood
- on small datasets (like ours), the compact PCA trick is fast and perfectly fine
- on large datasets, SVD scales better and is the industry default